# EEG Mental Workload Classifier — Phase 3: Deep Learning, Explainability, Deployment Prep

**What this notebook does, in order:**
1. Load checkpoints from Notebook 2 (raw windows + engineered features)
2. Train a 1D-CNN directly on raw filtered/cleaned EEG windows, with subject-independent CV, and compare to the Notebook 2 baseline
3. Run SHAP explainability on the Random Forest baseline -- which bands/channels actually drive predictions
4. Decide 3-class vs. binary framing for the deployed demo
5. Save the final chosen model for deployment

**Read the markdown before each code cell** -- same practice as Notebooks 1-2: understand what a step does and what "working" looks like before trusting its output.

## 0. Setup + load checkpoints

In [ ]:
!pip install -q torch scikit-learn shap matplotlib seaborn pandas

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/eeg-workload-project'
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

sns.set_theme(style='whitegrid')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

SFREQ = 128
CHANNEL_NAMES = ['AF3','F7','F3','FC5','T7','P7','O1','O2','P8','T8','FC6','F4','F8','AF4']

X_windows = np.load(f'{PROJECT_DIR}/data/processed/X_windows.npy')       # (n_windows, 14, 512) raw filtered+cleaned
X_features = np.load(f'{PROJECT_DIR}/data/processed/X_features.npy')     # (n_windows, 56) band-power features
y_windows = np.load(f'{PROJECT_DIR}/data/processed/y_windows.npy')
subj_windows = np.load(f'{PROJECT_DIR}/data/processed/subj_windows.npy')
with open(f'{PROJECT_DIR}/data/processed/feature_names.txt') as f:
    feature_names = f.read().splitlines()

print('X_windows:', X_windows.shape, '| X_features:', X_features.shape, '| y_windows:', y_windows.shape)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## 1. 1D-CNN on raw windows

**What this does:** instead of hand-computing band-power features, we let a convolutional network learn its own features directly from the raw 512-timepoint signal per channel. This answers a real question: does the theory-driven feature engineering from Notebook 2 capture most of the useful signal, or is a learned representation meaningfully better?

**Why this comparison matters for your project story:** it's not about "deep learning beats classical ML" (it often doesn't, especially on small datasets) -- it's about being able to say *which* approach worked better here and *why*, which is a more sophisticated claim than defaulting to whichever method is trendier.

**Architecture choice:** a small 1D-CNN (not a large one) is appropriate here -- with only 45 subjects, a large model will overfit almost immediately. We keep the network shallow and add dropout deliberately.

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class EEG1DCNN(nn.Module):
    """Small 1D-CNN: two conv blocks + global average pooling + linear head.
    Deliberately shallow -- with only 45 subjects, a deep network will overfit
    almost immediately, so we prioritize a small parameter count over capacity."""
    def __init__(self, n_channels=14, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(4),
            nn.Dropout(0.3),

            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),  # global average pooling -- collapses time axis
            nn.Dropout(0.3),
        )
        self.head = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.net(x)
        x = x.squeeze(-1)
        return self.head(x)

print(EEG1DCNN())

## 2. Train + evaluate with subject-independent CV

**Same principle as Notebook 2:** every fold's validation set contains only subjects never seen in that fold's training data. We reuse `GroupKFold` for consistency and a fair, direct comparison against the Section 6 baseline numbers (RF 0.375, SVM 0.470).

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score

N_SPLITS = 5
N_EPOCHS = 25
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

def train_one_fold(X_train, y_train, X_val, y_val, n_classes=3, n_epochs=N_EPOCHS, device=device):
    train_ds = EEGDataset(X_train, y_train)
    val_ds = EEGDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = EEG1DCNN(n_channels=X_train.shape[1], n_classes=n_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            preds = model(xb).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(yb.numpy())
    return accuracy_score(all_true, all_preds), model

gkf = GroupKFold(n_splits=N_SPLITS)
cnn_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_windows, y_windows, groups=subj_windows)):
    acc, _ = train_one_fold(X_windows[train_idx], y_windows[train_idx], X_windows[val_idx], y_windows[val_idx])
    cnn_scores.append(acc)
    print(f'Fold {fold}: accuracy = {acc:.3f}')

cnn_scores = np.array(cnn_scores)
print(f'\nCNN subject-independent: mean = {cnn_scores.mean():.3f} +/- {cnn_scores.std():.3f}')
print('Compare to Notebook 2 baseline: RF = 0.375, SVM = 0.470')

**How to read this.** Given only 45 subjects, don't expect the CNN to dramatically outperform the engineered-feature baseline -- deep learning generally needs much more data than classical ML with good features to show its advantage, and this is a well-known, explainable pattern in EEG ML specifically, not a sign anything is wrong. Three plausible, all-reportable outcomes:
- **CNN performs similarly to SVM (~0.45-0.50):** suggests the theory-driven band-power features already captured most of the useful signal -- a good finding, since it validates the feature engineering approach.
- **CNN underperforms the baseline:** expected at this sample size -- a legitimate, explainable result ("deep learning needs more data than we have to outperform domain-informed features here"), not a failure of the notebook.
- **CNN outperforms the baseline:** would be a genuinely interesting finding worth highlighting, but treat a large jump with some skepticism -- rerun to check it's not a lucky fold split, since small-n results can be noisy.

## 3. SHAP explainability on the Random Forest baseline

**What this does:** SHAP (SHapley Additive exPlanations) assigns each feature a contribution value for each individual prediction, based on cooperative game theory -- it answers "how much did this specific feature push this specific prediction toward or away from a given class," not just "is this feature generally important."

**Why this matters for your project:** this is the step that connects your model back to the psychology/neuroscience framing concretely. If SHAP shows theta and alpha features (especially frontal/parietal ones) among the most influential, that's independent, model-derived evidence supporting the cognitive-load-theory framing -- not just something we assumed going in.

In [ ]:
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Train one RF on a subject-independent train/test split (not full CV) --
# SHAP explains a specific trained model's behavior, so we need one fixed model.
unique_subjects = np.unique(subj_windows)
train_subjects, test_subjects = train_test_split(unique_subjects, test_size=0.25, random_state=RANDOM_SEED)
train_mask = np.isin(subj_windows, train_subjects)
test_mask = np.isin(subj_windows, test_subjects)

rf_final = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, class_weight='balanced')
rf_final.fit(X_features[train_mask], y_windows[train_mask])
test_acc = rf_final.score(X_features[test_mask], y_windows[test_mask])
print(f'Held-out subject-independent test accuracy for this specific model: {test_acc:.3f}')

explainer = shap.TreeExplainer(rf_final)
# Use a subsample of the test set for speed -- SHAP on tree ensembles is exact
# but can be slow on thousands of rows.
sample_idx = np.random.choice(np.where(test_mask)[0], size=min(300, test_mask.sum()), replace=False)
shap_values = explainer.shap_values(X_features[sample_idx])

print('SHAP values computed. Shape info:', np.array(shap_values).shape if isinstance(shap_values, list) else shap_values.shape)

In [ ]:
# Summary plot: which features matter most overall, averaged across all three classes
shap.summary_plot(shap_values, X_features[sample_idx], feature_names=feature_names, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/figures/shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()

**How to read this.** Look at which band shows up most among the top features -- theta and alpha features appearing prominently (especially from frontal or parietal/occipital channels) would support the cognitive-load-theory framing directly. If beta/gamma or unexpected channels dominate instead, that's still a valid, reportable finding -- just a different one than the textbook prediction, and worth noting honestly rather than only reporting results that confirm the theory.

## 4. Decide: 3-class or binary for the deployed demo

**The tradeoff:** the 3-class task (low/moderate/high) is what the dataset provides natively and what we've evaluated so far. Collapsing to binary (low vs. high, dropping moderate) is the standard published STEW framing and typically yields higher, more stable accuracy, since removing the ambiguous middle category removes the hardest-to-separate boundary cases.

**Why this matters for the demo specifically:** a deployed interactive app benefits from a model that gives confident, defensible predictions -- a binary framing may make for a cleaner, more convincing live demo even if the 3-class analysis remains in your notebooks as the more complete, rigorous evaluation.

In [ ]:
# Quick check: how does the same RF pipeline perform if we drop the 'moderate'
# class entirely and treat this as binary low-vs-high?
binary_mask = y_windows != 1  # drop moderate (label 1)
X_binary = X_features[binary_mask]
y_binary = (y_windows[binary_mask] == 2).astype(int)  # 0=low, 1=high
subj_binary = subj_windows[binary_mask]

from sklearn.model_selection import cross_val_score
gkf_binary = GroupKFold(n_splits=N_SPLITS)
rf_binary_scores = cross_val_score(
    RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, class_weight='balanced'),
    X_binary, y_binary, cv=gkf_binary, groups=subj_binary, scoring='accuracy'
)
print(f'Binary (low vs. high) RF, subject-independent: mean = {rf_binary_scores.mean():.3f} +/- {rf_binary_scores.std():.3f}')
print(f'Binary chance level: 0.500')
print(f'\nFor comparison, 3-class RF was: mean = 0.375, chance = 0.333')

**Decide based on the numbers above.** If binary accuracy clears its 0.5 chance level by a wider margin than 3-class cleared its 0.333 chance level (i.e. binary is relatively more confident, not just numerically higher against an easier baseline), that supports using binary for the deployed demo. Either choice is defensible -- document whichever you pick and why in your project write-up; showing you considered the tradeoff deliberately is itself a good signal.

## 5. Save final model + preprocessing artifacts for deployment

**What this does:** persists whichever model you decide to deploy (RF, SVM, or CNN), plus everything needed to reproduce the exact preprocessing pipeline on a brand-new raw EEG sample at inference time -- the deployment app in Notebook 4 will need to bandpass-filter, run the same ICA logic, extract the same band-power features (if using the RF/SVM path), and only then call the model.

**Fill in `CHOSEN_FRAMING` and `CHOSEN_MODEL` based on your decision above before running this cell.**

In [ ]:
import joblib

CHOSEN_FRAMING = '3-class'  # or 'binary' -- set based on Section 4's results
CHOSEN_MODEL = 'RandomForest'  # or 'SVM' or 'CNN'

if CHOSEN_MODEL in ('RandomForest', 'SVM'):
    # Retrain the chosen model on ALL available data (not just the SHAP train split)
    # for the strongest possible deployed model, now that evaluation is done.
    if CHOSEN_FRAMING == '3-class':
        X_final, y_final = X_features, y_windows
    else:
        X_final, y_final = X_binary, y_binary

    if CHOSEN_MODEL == 'RandomForest':
        final_model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, class_weight='balanced')
    else:
        from sklearn.svm import SVC
        from sklearn.preprocessing import StandardScaler
        from sklearn.pipeline import Pipeline
        final_model = Pipeline([('scaler', StandardScaler()), ('svm', SVC(kernel='rbf', class_weight='balanced', probability=True))])

    final_model.fit(X_final, y_final)
    joblib.dump(final_model, f'{PROJECT_DIR}/models/final_model.joblib')
    print(f'Saved {CHOSEN_MODEL} ({CHOSEN_FRAMING}) to models/final_model.joblib')
else:
    print('CNN chosen -- retrain on full data and torch.save() the state_dict separately before deployment.')

# Save metadata the deployment app will need regardless of model choice
import json
metadata = {
    'framing': CHOSEN_FRAMING,
    'model_type': CHOSEN_MODEL,
    'feature_names': feature_names,
    'channel_names': CHANNEL_NAMES,
    'sfreq': SFREQ,
    'window_seconds': 4,
    'bands': {'theta': [4, 8], 'alpha': [8, 13], 'beta': [13, 30], 'gamma': [30, 45]},
}
with open(f'{PROJECT_DIR}/models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved deployment metadata.')

## Next steps (Notebook 4)

1. Build the Streamlit/Gradio app: user selects or uploads a sample EEG window, app runs the same preprocessing pipeline, calls the saved model, and displays the prediction alongside a SHAP explanation for that specific input
2. Deploy to Hugging Face Spaces or Streamlit Community Cloud
3. Write the project README summarizing the pipeline, the leakage-correction finding, the normalization negative result, and the SHAP-derived theory validation -- these are your strongest, most specific talking points